In [6]:
import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
import json

In [7]:
%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Python314\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\nerii\Desktop\Work\DVS\Manson Maze\py\env\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "d:\nerii\Desktop\Work\DVS\Manson Maze\py\env\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Python314\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1398, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1335, in _find_and_load_unlocked
ModuleNotFoundError: No module named 'autoreload'

During handling of the ab

In [12]:
from utils.navigation import Agent, Maze, explore_maze

Failed to read module file 'd:\nerii\Desktop\Work\DVS\Manson Maze\py\utils\navigation.py' for module 'utils.navigation': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\nerii\Desktop\Work\DVS\Manson Maze\py\env\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 556, in maybe_reload_module
    new_source_code = f.read()
  File "C:\Python314\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 2750: character maps to <undefined>


# Settings

In [13]:
test = "gemini-3-flash-preview"
iterations = 25


output_dir = os.path.join("outputs", test)
os.makedirs(output_dir, exist_ok=True)

# Run Exploration

In [14]:
model = ChatGoogleGenerativeAI(model=test, request_timeout=180, max_retries=3)

# Prepare data structure and load existing data if available
params = ["travel_logs", "decision_logs", "analysis_logs", "last_notes", "end_causes"]

data = {}
for key in params:
    file_path = os.path.join(output_dir, f"{key}.json")
    
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data[key] = json.load(f)
    else:
        data[key] = []

In [ ]:
initial_iteration = len(data["travel_logs"]) 
last_iteration = initial_iteration + iterations


for iter in range(initial_iteration, last_iteration):

    print( f"\n--- Iteration {iter} ---\n")
        
    maze = Maze()
    agent = Agent(model=model)

    # Inject past notes
    if len(data["last_notes"]) > 0:
        agent.last_notes = "\n".join(f"Attempt {note['iteration']}: {note['data']}" for note in data["last_notes"])

    # Run Iteration
    new_data, agent = explore_maze(agent, maze, max_steps=32)

    print("Causes of failure:", agent.status, "/ note:", new_data["last_notes"])

    # Store outputs for next run
    for key, new_items in new_data.items():
        if key in data:
            data[key].append({"iteration": iter, "data": new_items})
        else:
            data[key] = [{"iteration": iter, "data": new_items}]

    # Save data to file
    for key, content in data.items():
        filename = os.path.join(output_dir, f"{key}.json")
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(content, f, indent=4)



--- Iteration 0 ---

AI Ready: True
Step 1: Moved from Room 1 to Room 20 ➡️
Step 2: Moved from Room 20 to Room 5 ➡️
Step 3: Moved from Room 5 to Room 30 ➡️
Step 4: Moved from Room 30 to Room 15 ➡️
Step 5: Moved from Room 15 to Room 3 ➡️
Hallucination detected: Room 37 is not a valid exit from Room 3.
Step 6: Moved from Room 3 to Room 9 ➡️
Step 7: Moved from Room 9 to Room 18 ➡️
Hallucination detected: Room 45 is not a valid exit from Room 18.
Hallucination detected: Room 45 is not a valid exit from Room 18.
Step 8: Moved from Room 18 to Room 13 ➡️
Step 9: Moved from Room 13 to Room 27 ➡️
Hallucination detected: Room 36 is not a valid exit from Room 27.
Hallucination detected: Room 34 is not a valid exit from Room 27.
Hallucination detected: Room 31 is not a valid exit from Room 27.
Hallucination detected: Room 35 is not a valid exit from Room 27.
Max attempts reached. Agent is hallucinated
Causes of failure: hallucinated / note: In Room 27, I was misled by the 'catch up' pun and the h